Siphon, a library used to access THREDDS, a data server that provides web-based access to scientific datasets and allows specific data extraction. (So you don't have to download everything needlessley)

In [ ]:
%pip install siphon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 kB 4.0 MB/s eta 0:00:00


Here we get and list available forecast files from 2015-01-15, The whole dataset for GFS 0.25 spans from 2015-01-15 to present

In [ ]:
from siphon.catalog import TDSCatalog
catalog = TDSCatalog('https://thredds.rda.ucar.edu/thredds/catalog/files/g/d084001/2015/20150115/catalog.xml')
print(catalog.datasets)

['gfs.0p25.2015011500.f000.grib2', 'gfs.0p25.2015011500.f003.grib2', 'gfs.0p25.2015011500.f006.grib2', 'gfs.0p25.2015011500.f009.grib2', 'gfs.0p25.2015011500.f012.grib2', 'gfs.0p25.2015011500.f015.grib2', 'gfs.0p25.2015011500.f018.grib2', 'gfs.0p25.2015011500.f021.grib2', 'gfs.0p25.2015011500.f024.grib2', 'gfs.0p25.2015011500.f027.grib2', 'gfs.0p25.2015011500.f030.grib2', 'gfs.0p25.2015011500.f033.grib2', 'gfs.0p25.2015011500.f036.grib2', 'gfs.0p25.2015011500.f039.grib2', 'gfs.0p25.2015011500.f042.grib2', 'gfs.0p25.2015011500.f045.grib2', 'gfs.0p25.2015011500.f048.grib2', 'gfs.0p25.2015011500.f051.grib2', 'gfs.0p25.2015011500.f054.grib2', 'gfs.0p25.2015011500.f057.grib2', 'gfs.0p25.2015011500.f060.grib2', 'gfs.0p25.2015011500.f063.grib2', 'gfs.0p25.2015011500.f066.grib2', 'gfs.0p25.2015011500.f069.grib2', 'gfs.0p25.2015011500.f072.grib2', 'gfs.0p25.2015011500.f075.grib2', 'gfs.0p25.2015011500.f078.grib2', 'gfs.0p25.2015011500.f081.grib2', 'gfs.0p25.2015011500.f084.grib2', 'gfs.0p25.201

**Understanding the file naming convention:**

`gfs.0p25.YYYYMMDDHH.fXXX.grib2`

Breaking It Down:

`gfs.0p25` → This refers to the GFS(Global Forecast System) model with 0.25-degree resolution (high resolution).

`YYYYMMDDHH` → The initialization time of the forecast in UTC. (REAL OBSERVATION STARTING POINT)

2015011500 → January 15, 2015, at 00:00 UTC. (02:00 SWEDISH SUMMER TIME)

2015011506 → January 15, 2015, at 06:00 UTC. (08:00 SWEDISH SUMMER TIME)

2015011512 → January 15, 2015, at 12:00 UTC. (14:00 SWEDISH SUMMER TIME)

2015011518 → January 15, 2015, at 18:00 UTC. (20:00 SWEDISH SUMMER TIME)

`fXXX` → The forecast hour.

f000 → This is the analysis time (the actual observations at initialization time).

f003 → 3-hour forecast from initialization.

f006 → 6-hour forecast from initialization.

f009 → 9-hour forecast from initialization.

…and so on, in 3-hour increments.

So, Let's extract the datasets spanning a 10 days prognosis, Meaning 240 hours onwards and start at 12UTC initialization.

In [ ]:
# List all datasets
all_datasets = list(catalog.datasets.keys())

init_time = "2015011512"  # (12 UTC) init
prognosis = 168 #forecast hours <= 240 (interval of 3 hours) max is 384 (16 days)
# Filter datasets that have "2015011512" (12 UTC) and forecast hours <= 240
filtered_files = [f for f in all_datasets if init_time in f and int(f.split(".f")[-1][:3]) <= prognosis]

# Print the selected files
print("Selected files for processing:", filtered_files)

Selected files for processing: ['gfs.0p25.2015011512.f000.grib2', 'gfs.0p25.2015011512.f003.grib2', 'gfs.0p25.2015011512.f006.grib2', 'gfs.0p25.2015011512.f009.grib2', 'gfs.0p25.2015011512.f012.grib2', 'gfs.0p25.2015011512.f015.grib2', 'gfs.0p25.2015011512.f018.grib2', 'gfs.0p25.2015011512.f021.grib2', 'gfs.0p25.2015011512.f024.grib2', 'gfs.0p25.2015011512.f027.grib2', 'gfs.0p25.2015011512.f030.grib2', 'gfs.0p25.2015011512.f033.grib2', 'gfs.0p25.2015011512.f036.grib2', 'gfs.0p25.2015011512.f039.grib2', 'gfs.0p25.2015011512.f042.grib2', 'gfs.0p25.2015011512.f045.grib2', 'gfs.0p25.2015011512.f048.grib2', 'gfs.0p25.2015011512.f051.grib2', 'gfs.0p25.2015011512.f054.grib2', 'gfs.0p25.2015011512.f057.grib2', 'gfs.0p25.2015011512.f060.grib2', 'gfs.0p25.2015011512.f063.grib2', 'gfs.0p25.2015011512.f066.grib2', 'gfs.0p25.2015011512.f069.grib2', 'gfs.0p25.2015011512.f072.grib2', 'gfs.0p25.2015011512.f075.grib2', 'gfs.0p25.2015011512.f078.grib2', 'gfs.0p25.2015011512.f081.grib2', 'gfs.0p25.201501

siphon and THREDDS allows us to query for subset regions, So let's define a subset region for Sweden

In [ ]:
# Define Sweden's bounding box
lat_min, lat_max = 55.0, 70.0
lon_min, lon_max = 10.0, 25.0

Now we need to download each subset for each file separately and then combine each dataset alongside time coord

Temperature_surface

Wind_speed_gust_surface

Relative_humidity_sigma

Relative_humidity_height_above_ground

Volumetric_Soil_Moisture_Content_depth_below_surface_layer

Soil_temperature_depth_below_surface_layer

Total_precipitation_surface_3_Hour_Accumulation


In [ ]:
from datetime import datetime
import xarray as xr
import pandas as pd
from io import BytesIO
import time

ds_list = []

#graderingstilfallen
#graderingsdatum
#

for file in filtered_files:
    dataset = catalog.datasets[file]

    # Check if NCSS is available
    if 'NetcdfSubset' in dataset.access_urls:

        ncss = dataset.subset()
        #print("Available variables:", ncss.variables)

        query = ncss.query()
        # Subset query for Sweden and the Temperature_surface variable
        query.lonlat_box(north=lat_max, south=lat_min, east=lon_max, west=lon_min)
        try:
          try:
            query.variables('Total_precipitation_surface_3_Hour_Accumulation')
            query.accept('netcdf4')


            print(f"Fetching data from {file}...")

            #First doesnt have any TOtal_precipitation_surface_3_hour_acccum how to skip?
            data = ncss.get_data(query)
            ds = xr.open_dataset(BytesIO(data))
          # Some has time1 instead of time? no idea why
            if 'time1' in ds.coords:
                ds = ds.rename({'time1': 'time'})

            ds_list.append(ds)
          except Exception as e:
            #print(e)
            pass
        except:
          pass



      #print(ds.variables)
      #time.sleep(99999)



# Now, concatenate all datasets along the time dimension.
if ds_list:
    ds_combined = xr.concat(ds_list, dim="time")
    print("Final dataset structure:")
    print(ds_combined)
else:
    print("No data was retrieved. Please check dataset availability.")

print(ds_combined.variables)


# Print a summary of the dataset
# print(ds)

# List all coordinate names
#print("Coordinates:", list(ds.coords))

# List all variable names
# print("Variables:", list(ds.variables))

#print(ds["reftime"].item())
#time.sleep(999999)

/usr/local/lib/python3.11/dist-packages/siphon/catalog.py:755: UserWarning: netCDF4 module not installed. Will be unable to handle NetCDF returns from NCSS.
  from .ncss import NCSS


Fetching data from gfs.0p25.2015011512.f000.grib2...
Fetching data from gfs.0p25.2015011512.f003.grib2...
Fetching data from gfs.0p25.2015011512.f006.grib2...
Fetching data from gfs.0p25.2015011512.f009.grib2...
Fetching data from gfs.0p25.2015011512.f012.grib2...
Fetching data from gfs.0p25.2015011512.f015.grib2...
Fetching data from gfs.0p25.2015011512.f018.grib2...
Fetching data from gfs.0p25.2015011512.f021.grib2...
Fetching data from gfs.0p25.2015011512.f024.grib2...
Fetching data from gfs.0p25.2015011512.f027.grib2...
Fetching data from gfs.0p25.2015011512.f030.grib2...
Fetching data from gfs.0p25.2015011512.f033.grib2...
Fetching data from gfs.0p25.2015011512.f036.grib2...
Fetching data from gfs.0p25.2015011512.f039.grib2...
Fetching data from gfs.0p25.2015011512.f042.grib2...
Fetching data from gfs.0p25.2015011512.f045.grib2...
Fetching data from gfs.0p25.2015011512.f048.grib2...
Fetching data from gfs.0p25.2015011512.f051.grib2...
Fetching data from gfs.0p25.2015011512.f054.gr

In [ ]:
import numpy as np
# Replace NaT values in time_bounds with corresponding values from time1_bounds
ds_combined['time_bounds'] = ds_combined['time_bounds'].where(
    ~np.isnat(ds_combined['time_bounds']),
    ds_combined['time1_bounds']
)

# Optionally, drop time1_bounds if no longer needed
ds_combined = ds_combined.drop_vars('time1_bounds')

output_filename = "gfs_sweden_regn.nc"
ds_combined.to_netcdf(output_filename)
print(f"Saved dataset to {output_filename}")

Saved dataset to gfs_sweden_regn.nc


Input target lat&lon and get back JSON file with values

In [ ]:
import pandas as pd
import json
# Define your target latitude and longitude
#öland
target_lat = 56.7  # replace with your specific latitude
target_lon = 16.6  # replace with your specific longitude

#select the nearest grid point using .sel()
point_data = ds_combined.sel(latitude=target_lat, longitude=target_lon, method="nearest")

#extract the time and precipitation arrays
time_values = point_data["time"].values
precip_values = point_data["Total_precipitation_surface_3_Hour_Accumulation"].values

time_precip_list = []
for t, precip in zip(time_values, precip_values):
    #Convert numpy.datetime64 to a formatted string
    time_str = pd.to_datetime(t).strftime("%Y-%m-%dT%H:%M:%SZ")
    time_precip_list.append({"time": time_str, "precipitation": float(precip)})

#Convert the list to a JSON string
json_output = json.dumps(time_precip_list, indent=2)
print(json_output)

[
  {
    "time": "2015-01-15T13:30:00Z",
    "precipitation": 0.30000001192092896
  },
  {
    "time": "2015-01-15T19:30:00Z",
    "precipitation": 1.7999999523162842
  },
  {
    "time": "2015-01-16T01:30:00Z",
    "precipitation": 0.30000001192092896
  },
  {
    "time": "2015-01-16T07:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-16T13:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-16T19:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-17T01:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-17T07:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-17T13:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-17T19:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-18T01:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-18T07:30:00Z",
    "precipitation": 0.699999988079071
  },
  {
    "time": "2015-01-18T13:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-18

here we will download all forecasted rain data 7 days back from all unique graderingstillfällen for Höstvete & Bladfläcksvampar

In [ ]:
from urllib.error import HTTPError
from siphon.catalog import TDSCatalog
from datetime import datetime, timedelta
import json
import requests
import xarray as xr
from io import BytesIO

with open("unique_dates.json","r") as f:
  data = json.load(f)



#go 7 days back from the graderingstillfallen
counter = 0
#found = False
for dateStr in data: #dateStr will be used as key in final json dictionary
  counter += 1
  #if dateStr == "2018-05-27" and not found:
    #found=True
  if counter >= 1:#found: #counter > 1: #used for skipping files(debugging)
    format = '%Y-%m-%d'
    dateObject = datetime.strptime(dateStr, format)

    dateObject = dateObject - timedelta(days=7)

    yearStr = dateObject.strftime("%Y")
    dateStr7DaysBack = dateObject.strftime("%Y%m%d")

    catalog = TDSCatalog(f'https://thredds.rda.ucar.edu/thredds/catalog/files/g/d084001/{yearStr}/{dateStr7DaysBack}/catalog.xml')
    #print(catalog.datasets)


    # List all datasets
    all_datasets = list(catalog.datasets.keys())


    init_time = dateStr7DaysBack + "12"
    prognosis = 168 #7 days in hours
    # Filter datasets that have "2015011512" (12 UTC) and forecast hours <= 240
    filtered_files = [f for f in all_datasets if init_time in f and int(f.split(".f")[-1][:3]) <= prognosis]

    # Print the selected files
    print("Selected files for processing:", filtered_files)

    ds_list = []


    #SKIP EVERYother FOR 3 HOUR AVG (RAIN)
    filtered_files = [file for i, file in enumerate(filtered_files) if i % 2 != 0]

    for file in filtered_files:
        dataset = catalog.datasets[file]

        # Check if NCSS is available
        if 'NetcdfSubset' in dataset.access_urls:

            ncss = dataset.subset()
            #print("Available variables:", ncss.variables)
            #time.sleep(9999999)


            query = ncss.query()
            # Subset query for Sweden and the Temperature_surface variable
            query.lonlat_box(north=lat_max, south=lat_min, east=lon_max, west=lon_min)

            try:
              #query.variables('Precipitation_rate_surface')
              query.variables('Categorical_Rain_surface_3_Hour_Average')
              query.accept('netcdf4')



              print(f"Fetching data from {file}...")

              #First doesnt have any TOtal_precipitation_surface_3_hour_acccum how to skip?
              data = ncss.get_data(query)
              ds = xr.open_dataset(BytesIO(data))
            # Some has time1 instead of time? no idea why
              if 'time1' in ds.coords:
                  ds = ds.rename({'time1': 'time'})
              if 'time2' in ds.coords:
                  ds = ds.rename({'time2': 'time'})

              #print(ds.coords)

              ds_list.append(ds)

            except requests.exceptions.HTTPError as err:
              #print("Server Error ( 400:" in str(err))
              print(err)
            except Exception as e:
              print(e)



    import numpy as np

    print(len(ds_list)) #1
    #forgot to combine opsi
    if ds_list:
        ds_combined = xr.concat(ds_list, dim="time")
        print("Final dataset structure:")
        #print(ds_combined)
    else:
        print("No data was retrieved. Please check dataset availability.")


    output_filename = f"drive/MyDrive/MLProject/weather/precipitation/gfs_sweden_precipitation_{dateStr}.nc"
    ds_combined.to_netcdf(output_filename)
    print(f"Saved dataset to {output_filename}")




Selected files for processing: ['gfs.0p25.2020062212.f000.grib2', 'gfs.0p25.2020062212.f003.grib2', 'gfs.0p25.2020062212.f006.grib2', 'gfs.0p25.2020062212.f009.grib2', 'gfs.0p25.2020062212.f012.grib2', 'gfs.0p25.2020062212.f015.grib2', 'gfs.0p25.2020062212.f018.grib2', 'gfs.0p25.2020062212.f021.grib2', 'gfs.0p25.2020062212.f024.grib2', 'gfs.0p25.2020062212.f027.grib2', 'gfs.0p25.2020062212.f030.grib2', 'gfs.0p25.2020062212.f033.grib2', 'gfs.0p25.2020062212.f036.grib2', 'gfs.0p25.2020062212.f039.grib2', 'gfs.0p25.2020062212.f042.grib2', 'gfs.0p25.2020062212.f045.grib2', 'gfs.0p25.2020062212.f048.grib2', 'gfs.0p25.2020062212.f051.grib2', 'gfs.0p25.2020062212.f054.grib2', 'gfs.0p25.2020062212.f057.grib2', 'gfs.0p25.2020062212.f060.grib2', 'gfs.0p25.2020062212.f063.grib2', 'gfs.0p25.2020062212.f066.grib2', 'gfs.0p25.2020062212.f069.grib2', 'gfs.0p25.2020062212.f072.grib2', 'gfs.0p25.2020062212.f075.grib2', 'gfs.0p25.2020062212.f078.grib2', 'gfs.0p25.2020062212.f081.grib2', 'gfs.0p25.202006

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Lets try running this with proxies instead & concurrent threads

In [ ]:
import json
import random
import requests
from siphon.catalog import TDSCatalog
from siphon.ncss import NCSS
from datetime import datetime, timedelta
import xarray as xr
from io import BytesIO
import concurrent.futures
import time
import requests

# Load unique dates (assumed to be a list of date strings, e.g. ["2018-05-27", ...])
with open("unique_dates.json", "r") as f:
    dates = json.load(f)

"""
tmp = []
found = False
for date in dates:
  if date == "2015-05-18" and not found:
    found = True
  if found == True:
    tmp.append(date)

dates = tmp
"""

# Load proxies from JSON (adjust the filename/path as needed)
with open("proxies.json", "r") as f:
    proxies_data = json.load(f)

# Build a list of proxy URLs using the first protocol listed for each proxy.
proxy_list = []
for proxy in proxies_data:
    protocol = proxy['protocols'][0]  # e.g., "socks5" or "socks4"
    ip = proxy['ip']
    port = proxy['port']
    proxy_url = f"{protocol}://{ip}:{port}"
    proxy_list.append(proxy_url)

print("Loaded proxies:", proxy_list)

# Define geographic bounds for the query over Sweden (modify as needed)
lat_min = 55.0
lat_max = 70.0
lon_min = 10.0
lon_max = 25.0

def download_dataset(file, catalog, proxy_list, lat_min, lat_max, lon_min, lon_max):
    """
    Downloads the dataset for the given file from the catalog using a randomly selected proxy.
    This function will keep retrying (with exponential backoff) until the data is successfully fetched.
    Returns an xarray.Dataset once successful.
    """
    dataset = catalog.datasets[file]
    if 'NetcdfSubset' not in dataset.access_urls:
        print(f"Dataset {file} doesn't support NetcdfSubset.")
        return None

    attempt = 0
    while True:
        attempt += 1
        try:
            # Choose a random proxy from the list
            selected_proxy = random.choice(proxy_list)
            proxies = {
                'http': selected_proxy,
                'https': selected_proxy,
            }
            # Create a custom requests session with the proxy settings.
            session = requests.Session()
            session.proxies = proxies

            # Create an NCSS object and overwrite its session with our custom session.
            ncss_url = dataset.access_urls['NetcdfSubset']
            ncss = NCSS(ncss_url)
            ncss.session = session

            query = ncss.query()
            query.lonlat_box(north=lat_max, south=lat_min, east=lon_max, west=lon_min)
            query.variables('Categorical_Rain_surface_3_Hour_Average')
            query.accept('netcdf4')

            print(f"Fetching data from {file} using proxy {selected_proxy} (attempt {attempt})...")
            data = ncss.get_data(query)
            ds = xr.open_dataset(BytesIO(data))

            # Sometimes the time coordinate is misnamed; rename if needed.
            if 'time1' in ds.coords:
                ds = ds.rename({'time1': 'time'})
            if 'time2' in ds.coords:
                ds = ds.rename({'time2': 'time'})

            return ds


        except Exception as e:
            if "Server Error ( 400:" in str(e):
              return None

            print(f"Error fetching {file} (attempt {attempt}): {e}")
            # Exponential backoff with a cap to avoid overly long sleeps
            sleep_time = 2 ** min(attempt, 5)
            #print(f"Retrying in {sleep_time} seconds...")
            time.sleep(sleep_time)

# Loop over each date in your unique_dates list.
for dateStr in dates:
    try:
        # Subtract 7 days from the given date.
        format_str = '%Y-%m-%d'
        date_obj = datetime.strptime(dateStr, format_str) - timedelta(days=7)
        yearStr = date_obj.strftime("%Y")
        dateStr7DaysBack = date_obj.strftime("%Y%m%d")
    except Exception as e:
        print(f"Error processing date {dateStr}: {e}")
        continue

    # Construct the catalog URL.
    catalog_url = f'https://thredds.rda.ucar.edu/thredds/catalog/files/g/d084001/{yearStr}/{dateStr7DaysBack}/catalog.xml'
    print(f"\nProcessing date {dateStr} using catalog: {catalog_url}")

    try:
        catalog = TDSCatalog(catalog_url)
    except Exception as e:
        print(f"Error loading catalog for {dateStr}: {e}")
        continue

    # List and filter datasets.
    all_datasets = list(catalog.datasets.keys())
    init_time = dateStr7DaysBack + "12"
    prognosis = 168  # 7 days in hours
    filtered_files = [f for f in all_datasets if init_time in f and int(f.split(".f")[-1][:3]) <= prognosis]

    #SKIP EVERYother FOR 3 HOUR AVG (RAIN)
    filtered_files = [file for i, file in enumerate(filtered_files) if i % 2 != 0]

    print("Selected files for processing:", filtered_files)
    ds_list = []

    # Use ThreadPoolExecutor to download datasets concurrently (max 10 workers).
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        # Map each file to a concurrent download.
        future_to_file = {
            executor.submit(download_dataset, file, catalog, proxy_list, lat_min, lat_max, lon_min, lon_max): file
            for file in filtered_files
        }
        for future in concurrent.futures.as_completed(future_to_file):
            file = future_to_file[future]
            try:
                ds = future.result()
                if ds is not None:
                    ds_list.append(ds)
            except Exception as exc:
                print(f"{file} generated an exception: {exc}")

    print("Number of datasets downloaded:", len(ds_list))
    if ds_list:
        try:
            # Concatenate along the time dimension.
            ds_combined = xr.concat(ds_list, dim="time")
            print("Final dataset structure:")
            #print(ds_combined)

            output_filename = f"drive/MyDrive/MLProject/weather/precipitation/gfs_sweden_precipitation_{dateStr}.nc"
            ds_combined.to_netcdf(output_filename)
            print(f"Saved dataset to {output_filename}")
        except Exception as e:
            print(f"Error concatenating or saving dataset for {dateStr}: {e}")
    else:
        print("No data was retrieved for this date. Please check dataset availability.")


Utdata för streaming har trunkerats till de sista 5000 raderna.
Fetching data from gfs.0p25.2020060812.f135.grib2 using proxy socks4://187.243.253.182:43015 (attempt 1)...
Fetching data from gfs.0p25.2020060812.f147.grib2 using proxy socks4://202.154.178.243:5678 (attempt 1)...
Fetching data from gfs.0p25.2020060812.f159.grib2 using proxy socks4://45.112.125.51:4145 (attempt 1)...
Fetching data from gfs.0p25.2020060812.f153.grib2 using proxy socks5://110.42.188.54:2080 (attempt 1)...
Fetching data from gfs.0p25.2020060812.f165.grib2 using proxy socks5://106.14.105.216:22016 (attempt 1)...
Number of datasets downloaded: 28
Final dataset structure:
Saved dataset to drive/MyDrive/MLProject/weather/precipitation/gfs_sweden_precipitation_2020-06-15.nc

Processing date 2016-06-27 using catalog: https://thredds.rda.ucar.edu/thredds/catalog/files/g/d084001/2016/20160620/catalog.xml
Selected files for processing: ['gfs.0p25.2016062012.f003.grib2', 'gfs.0p25.2016062012.f009.grib2', 'gfs.0p25.201